# MathQuest - Results Analysis

Exploratory analysis of user interaction data collected during the prototype evaluation.

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "font.size": 11,
        "axes.titlesize": 12,
        "axes.labelsize": 11,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.constrained_layout.use": True,
    }
)

DIFF_COLORS = {"easy": "#4CAF50", "medium": "#FFC107", "hard": "#F44336"}
ZPD_COLORS = {"Below ZPD": "#4CAF50", "Within ZPD": "#FFC107", "Above ZPD": "#F44336"}
DIFF_KEYS = ["easy", "medium", "hard"]

: 

In [ ]:
df = pd.read_csv("users_data.csv")

survey_q = ["q1", "q2", "q3", "q4", "q5", "age"]
df[survey_q] = df[survey_q].replace("", np.nan).apply(pd.to_numeric, errors="coerce")

for d in DIFF_KEYS:
    df[f"{d}_accuracy"] = np.where(df[f"{d}_attempts"] > 0, df[f"{d}_correct"] / df[f"{d}_attempts"], np.nan)
    df[f"{d}_hint_rate"] = np.where(df[f"{d}_attempts"] > 0, df[f"{d}_hints"] / df[f"{d}_attempts"], np.nan)

df["total_attempts"] = df[[f"{d}_attempts" for d in DIFF_KEYS]].sum(axis=1)
df["total_correct"] = df[[f"{d}_correct" for d in DIFF_KEYS]].sum(axis=1)
df["total_hints"] = df[[f"{d}_hints" for d in DIFF_KEYS]].sum(axis=1)
df["overall_accuracy"] = np.where(df["total_attempts"] > 0, df["total_correct"] / df["total_attempts"], np.nan)

has_survey = df["q1"].notna()

print(f"Total users:       {len(df)}")
print(f"Users with survey: {has_survey.sum()}")
print(f'Total attempts:    {df["total_attempts"].sum():,.0f}')
print(f'Overall accuracy:  {df["overall_accuracy"].mean():.1%}')
print()

for d in DIFF_KEYS:
    n = (df[f"{d}_attempts"] > 0).sum()
    print(
        f"{d:8s}\tn={n:3d}\t"
        f'attempts={df[f"{d}_attempts"].sum()}\t'
        f'accuracy={df[f"{d}_accuracy"].mean():.1%}\t'
        f'hints/attempt={df[f"{d}_hint_rate"].mean():.2f}'
    )

---
## User Engagement

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution of total attempts per user
bins = [0, 5, 10, 20, 30, 50, 75, 100, int(df["total_attempts"].max()) + 1]
med = df["total_attempts"].median()

axes[0].hist(df["total_attempts"], bins=bins, color="#5C6BC0", edgecolor="white", linewidth=0.6)
axes[0].axvline(med, color="#F44336", linestyle="--", linewidth=1.5, label=f"Median = {med:.0f}")
axes[0].set_xticks(bins)
axes[0].set_xlabel("Total attempts per user")
axes[0].set_ylabel("Number of users")
axes[0].set_title("Distribution of user engagement")
axes[0].legend()

# Total attempts by difficulty
diff_totals = {d.capitalize(): df[f"{d}_attempts"].sum() for d in DIFF_KEYS}
colors = [DIFF_COLORS[d] for d in DIFF_KEYS]

bars = axes[1].bar(diff_totals.keys(), diff_totals.values(), color=colors, edgecolor="white", linewidth=0.6)
for bar, val in zip(bars, diff_totals.values()):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 3, str(int(val)), ha="center", va="bottom", fontsize=10
    )
axes[1].set_ylabel("Total attempts")
axes[1].set_title("Attempts by difficulty level")

plt.savefig("../figures/user_engagement.png", bbox_inches="tight")
plt.show()

---
## Performance by Difficulty

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

difficulties = [d.capitalize() for d in DIFF_KEYS]
acc_data = [df[f"{d}_accuracy"].dropna().values for d in DIFF_KEYS]
hint_data = [df[f"{d}_hint_rate"].dropna().values for d in DIFF_KEYS]

# Accuracy violin
parts = axes[0].violinplot(acc_data, positions=[1, 2, 3], showmedians=True, showextrema=False)
for pc, d in zip(parts["bodies"], DIFF_KEYS):
    pc.set_facecolor(DIFF_COLORS[d])
    pc.set_alpha(1.0)

parts["cmedians"].set_color("black")
parts["cmedians"].set_linewidth(2)

for i, vals in enumerate(acc_data):
    axes[0].scatter(i + 1, np.mean(vals), zorder=5, color="black", s=45, marker="D")

axes[0].set_xticks([1, 2, 3])
axes[0].set_xticklabels(difficulties)
axes[0].set_ylabel("Accuracy (correct / attempts)")
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[0].set_title("Accuracy distribution by difficulty")

# Hint rate bar
hint_means = [np.mean(v) for v in hint_data]
hint_sems = [np.std(v) / np.sqrt(len(v)) for v in hint_data]

bars = axes[1].bar(
    difficulties,
    hint_means,
    yerr=hint_sems,
    color=[DIFF_COLORS[d] for d in DIFF_KEYS],
    edgecolor="white",
    linewidth=0.8,
    capsize=5,
    error_kw={"linewidth": 1.5},
)
axes[1].set_ylabel("Hints per attempt")
axes[1].set_title("Hint usage by difficulty")

plt.savefig("../figures/user_performance.png", bbox_inches="tight")
plt.show()